In [1]:
import pandas as pd

# Load and Inspect Raw Data

In [2]:
orders = pd.read_csv(
    "../01_data/raw/olist_orders_dataset.csv"
)

items = pd.read_csv(
    "../01_data/raw/olist_order_items_dataset.csv"
)

products = pd.read_csv(
    "../01_data/raw/olist_products_dataset.csv"
)

sellers = pd.read_csv(
    "../01_data/raw/olist_sellers_dataset.csv"
)

customers = pd.read_csv(
    "../01_data/raw/olist_customers_dataset.csv"
)

payments = pd.read_csv(
    "../01_data/raw/olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    "../01_data/raw/olist_order_reviews_dataset.csv"
)

for name, df in {
    "orders": orders,
    "items": items,
    "products": products,
    "sellers": sellers,
    "customers": customers,
    "payments": payments,
    "reviews": reviews,
}.items():
    print(f"\n{name.upper()}")
    print(df.shape)
    display(df.info())


ORDERS
(99441, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

ITEMS
(112650, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_i

None

None

None

None

None

None

None

In [3]:
print("Orders")
print("Rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())
print("Unique customer IDs:", orders["customer_id"].nunique())

print("\nItems")
print("Rows:", len(items))
print("Unique order IDs:", items["order_id"].nunique())
print("Unique product IDs:", items["product_id"].nunique())
print("Unique seller IDs:", items["seller_id"].nunique())

print("\nProducts")
print("Rows:", len(products))
print("Unique product IDs:", products["product_id"].nunique())

print("\nSellers")
print("Rows:", len(sellers))
print("Unique seller IDs:", sellers["seller_id"].nunique())

print("\nCustomers")
print("Rows:", len(customers))
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Unique customer unique IDs:", customers["customer_unique_id"].nunique())

print("\nPayments")
print("Rows:", len(payments))
print("Unique order IDs:", payments["order_id"].nunique())

print("\nReviews")
print("Rows:", len(reviews))
print("Unique review IDs:", reviews["review_id"].nunique())
print("Unique order IDs:", reviews["order_id"].nunique())

Orders
Rows: 99441
Unique order IDs: 99441
Unique customer IDs: 99441

Items
Rows: 112650
Unique order IDs: 98666
Unique product IDs: 32951
Unique seller IDs: 3095

Products
Rows: 32951
Unique product IDs: 32951

Sellers
Rows: 3095
Unique seller IDs: 3095

Customers
Rows: 99441
Unique customer IDs: 99441
Unique customer unique IDs: 96096

Payments
Rows: 103886
Unique order IDs: 99440

Reviews
Rows: 99224
Unique review IDs: 98410
Unique order IDs: 98673


**Customer identifiers:** `customer_id` identifies the customer record associated
with an individual order, while `customer_unique_id` provides a persistent
identifier for the same underlying customer across multiple orders. Therefore,
`customer_unique_id` should be used when analyzing repeat purchasing or
customer-level behavior across orders.

In [4]:
items_per_order = items.groupby("order_id").size()
sellers_per_order = items.groupby("order_id")["seller_id"].nunique()
products_per_order = items.groupby("order_id")["product_id"].nunique()
payments_per_order = payments.groupby("order_id").size()
reviews_per_order = reviews.groupby("order_id").size()

print("Items per order")
display(items_per_order.describe())

print("\nSellers per order")
display(sellers_per_order.value_counts().sort_index())

print("\nProducts per order")
display(products_per_order.describe())

print("\nPayments per order")
display(payments_per_order.value_counts().sort_index())

print("\nReviews per order")
display(reviews_per_order.value_counts().sort_index())

Items per order

Sellers per order

Products per order

Payments per order

Reviews per order


count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

count    98666.000000
mean         1.038098
std          0.226456
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          8.000000
Name: product_id, dtype: float64

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

1    98126
2      543
3        4
Name: count, dtype: int64

# Relational Structure and Baseline Population

In [5]:
orders_without_items = orders.loc[
    ~orders["order_id"].isin(items["order_id"])
].copy()

print("Orders without item records:", len(orders_without_items))

display(
    orders_without_items["order_status"]
    .value_counts()
    .to_frame("orders")
)

Orders without item records: 775


,orders
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


In [6]:
orders_without_items_check = orders_without_items[
    ["order_id", "order_status", "customer_id"]
].copy()

orders_without_items_check["has_customer"] = (
    orders_without_items_check["customer_id"].isin(customers["customer_id"])
)

orders_without_items_check["has_payment"] = (
    orders_without_items_check["order_id"].isin(payments["order_id"])
)

orders_without_items_check["has_review"] = (
    orders_without_items_check["order_id"].isin(reviews["order_id"])
)

# Seller attribution requires an order-item record because seller_id
# is linked to order_id through the items table.
orders_without_items_check["has_seller_attribution"] = (
    orders_without_items_check["order_id"].isin(
        items.loc[items["seller_id"].notna(), "order_id"]
    )
)

display(
    orders_without_items_check.groupby("order_status")[
        [
            "has_customer",
            "has_payment",
            "has_review",
            "has_seller_attribution",
        ]
    ]
    .agg(["sum", "mean"])
    .round(3)
)

has_customer      has_payment  ... has_review has_seller_attribution     
                      sum mean         sum  ...       mean                    sum mean
order_status                                ...                                       
canceled              164  1.0         164  ...      0.982                      0  0.0
created                 5  1.0           5  ...      0.600                      0  0.0
invoiced                2  1.0           2  ...      1.000                      0  0.0
shipped                 1  1.0           1  ...      1.000                      0  0.0
unavailable           603  1.0         603  ...      0.977                      0  0.0

[5 rows x 8 columns]

The 775 orders without item records are primarily unavailable or canceled
orders. All have associated customer and payment records, and most also
have reviews. However, seller attribution is not available because
`seller_id` is linked to `order_id` through the order-item table.

Because the consolidated baseline uses the order-item table as its base,
these orders are naturally excluded. This is appropriate for seller-level
analysis because orders without item records cannot be attributed to a
seller.

# `Reviews` Structure

Reviews cannot be merged directly into the order-item baseline because neither
`order_id` nor `review_id` is unique in the reviews table. Before aggregating
reviews to the order level, examine the relationships between review records,
orders, and customers to determine whether repeated records represent duplicate
data or distinct review activity.

In [7]:
# Review records per order
reviews_per_order = reviews.groupby("order_id").size()
multi_review_orders = reviews_per_order[reviews_per_order > 1].index

multi_order_review_summary = (
    reviews[reviews["order_id"].isin(multi_review_orders)]
    .groupby("order_id")
    .agg(
        review_rows=("review_id", "size"),
        unique_review_ids=("review_id", "nunique"),
        unique_scores=("review_score", "nunique"),
    )
)

# Review IDs associated with multiple orders
review_id_summary = (
    reviews.groupby("review_id")
    .agg(
        review_rows=("review_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_scores=("review_score", "nunique"),
    )
)

shared_review_ids = review_id_summary[
    review_id_summary["unique_orders"] > 1
].index

shared_review_customers = (
    reviews[reviews["review_id"].isin(shared_review_ids)]
    .merge(
        orders[["order_id", "customer_id"]],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    .groupby("review_id")["customer_unique_id"]
    .nunique()
)

print("Orders with multiple review records:", len(multi_review_orders))
print(
    "Orders with multiple review scores:",
    (multi_order_review_summary["unique_scores"] > 1).sum(),
)
print(
    "Review IDs associated with multiple orders:",
    len(shared_review_ids),
)
print(
    "Shared review IDs belonging to one unique customer:",
    (shared_review_customers == 1).sum(),
)

Orders with multiple review records: 547
Orders with multiple review scores: 202
Review IDs associated with multiple orders: 789
Shared review IDs belonging to one unique customer: 789


The review table contains two distinct forms of repeated records:

- **547 orders have multiple review records.** Each has multiple distinct
  `review_id` values, and 202 contain more than one review score. These records
  therefore contain potentially meaningful review activity rather than simple
  duplicates.
- **789 review IDs are associated with multiple orders.** In every case, the
  associated orders belong to the same `customer_unique_id`, indicating that a
  single review event can be associated with multiple orders from the same
  underlying customer.

Therefore, neither `review_id` nor `order_id` should be treated as a unique key,
and review records should not be arbitrarily deduplicated. Reviews will instead
be summarized at the order level before being merged into the order-item
baseline.

# Date Conversion and Temporal Data Quality

In [8]:
review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp",
]

for col in review_date_cols:
    reviews[col] = pd.to_datetime(reviews[col])

order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders[order_date_cols] = orders[order_date_cols].apply(pd.to_datetime)

items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"]
)

In [9]:
anomalies = pd.Series({
    "approval_before_purchase": (
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"]
    ).sum(),

    "carrier_before_approval": (
        orders["order_delivered_carrier_date"]
        < orders["order_approved_at"]
    ).sum(),

    "delivery_before_carrier": (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum(),

    "delivery_before_purchase": (
        orders["order_delivered_customer_date"]
        < orders["order_purchase_timestamp"]
    ).sum(),
})

display(anomalies)

approval_before_purchase       0
carrier_before_approval     1359
delivery_before_carrier       23
delivery_before_purchase       0
dtype: int64

In [10]:
anomalous_carrier = orders.loc[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"],
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
].copy()

anomalous_carrier["carrier_before_approval_days"] = (
    anomalous_carrier["order_approved_at"]
    - anomalous_carrier["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

display(
    anomalous_carrier["carrier_before_approval_days"].describe()
)

count    1359.000000
mean        1.031333
std         4.804491
min         0.000243
25%         0.058976
50%         0.715324
75%         1.081528
max       171.219005
Name: carrier_before_approval_days, dtype: float64

Timestamp inconsistencies are retained as source-data limitations. Actual fulfillment and approval delays are not placement-time predictors. Recorded approval is not a reliable universal scoring gate; carrier-before-approval cases are quantified above. Invalid derived durations should be missing in descriptive comparisons.

## Review timing and retained anomalies

Review answers before purchase are suspicious; answers before delivery can represent real feedback on incomplete fulfillment. Eight item-attributable review records precede purchase, six sharing IDs with another order. They were reviewed and retained as negligible source anomalies; no new row exclusions are applied.

In [11]:
review_timing = reviews.merge(
    orders[
        ["order_id", "order_purchase_timestamp", "order_approved_at",
         "order_delivered_carrier_date", "order_delivered_customer_date"]
    ],
    on="order_id", how="left", validate="many_to_one",
)
review_timing["has_items"] = review_timing["order_id"].isin(items["order_id"])
timing_rows = []
for event in ["order_purchase_timestamp", "order_approved_at",
              "order_delivered_carrier_date", "order_delivered_customer_date"]:
    comparable = review_timing[event].notna()
    before = review_timing["review_answer_timestamp"].lt(review_timing[event])
    timing_rows.append(
        {"event": event, "comparable_reviews": comparable.sum(),
         "before_event_reviews": before.sum(),
         "before_event_orders": review_timing.loc[before, "order_id"].nunique()}
    )
display(pd.DataFrame(timing_rows))

pre_purchase_reviews = review_timing.loc[
    review_timing["has_items"]
    & review_timing["review_answer_timestamp"].lt(
        review_timing["order_purchase_timestamp"]
    )
].copy()
display(pre_purchase_reviews[
    ["order_id", "review_id", "review_score", "review_creation_date",
     "review_answer_timestamp", "order_purchase_timestamp"]
].sort_values("review_answer_timestamp"))

linked_reviews = review_timing.loc[
    review_timing["review_id"].isin(pre_purchase_reviews["review_id"])
]
display(linked_reviews[
    ["review_id", "order_id", "review_score", "review_answer_timestamp",
     "order_purchase_timestamp"]
].sort_values(["review_id", "order_purchase_timestamp"]))


,event,comparable_reviews,before_event_reviews,before_event_orders
0,order_purchase_timestamp,99224,63,63
1,order_approved_at,99068,20,20
2,order_delivered_carrier_date,97468,226,225
3,order_delivered_customer_date,96359,4795,4769


,order_id,review_id,review_score,review_creation_date,review_answer_timestamp,order_purchase_timestamp
92017,450c49623c365a4edcf0c5a2c93aa7c9,778826f9af6467ff6268bbcf436cd3c5,1,2017-02-22,2017-02-22 14:05:09,2017-03-01 08:08:10
24262,82fd1196a459f594fb1d66e667fc74c4,19194f53840fc6c2857dd8809972a209,4,2018-01-23,2018-01-24 12:43:58,2018-04-10 21:09:18
36453,af07804a2d53e8639954baf97fbf803a,a169c853a5ba10abfccb9c9afb06aa60,1,2018-03-10,2018-03-11 00:15:42,2018-04-19 13:46:42
24192,ebc94658c583ab37ad4f8e9091c4bef2,d25026282e14fc531f0711246e121cc2,2,2018-03-30,2018-04-02 00:32:35,2018-04-14 08:22:43
41449,96f5be02bc9ffc589f3274500a64a7e2,e2fc0692ed820a5802c1505a77cf7870,1,2018-04-04,2018-04-05 03:20:55,2018-04-23 14:31:55
62675,4a62fb19d5fa08fb514619dfcc617b3d,a935b3d67f406bac4853776d80a05220,3,2018-05-05,2018-05-08 02:06:50,2018-05-09 19:23:46
56320,4bb9c2002502ca416276dc1ff5efb1b3,0857872f03ef22e3dbe8391c4dd331f6,1,2018-05-18,2018-05-21 18:15:06,2018-06-23 14:53:03
78242,54282e97f61c23b78330c15b154c867d,6efce984242ca6456d74750810733a03,1,2018-08-31,2018-08-31 09:51:47,2018-09-03 09:06:57


,review_id,order_id,review_score,review_answer_timestamp,order_purchase_timestamp
56320,0857872f03ef22e3dbe8391c4dd331f6,4bb9c2002502ca416276dc1ff5efb1b3,1,2018-05-21 18:15:06,2018-06-23 14:53:03
60281,19194f53840fc6c2857dd8809972a209,ca263afd88a8a1200605adbd4b63cd7d,4,2018-01-24 12:43:58,2018-01-12 18:17:53
24262,19194f53840fc6c2857dd8809972a209,82fd1196a459f594fb1d66e667fc74c4,4,2018-01-24 12:43:58,2018-04-10 21:09:18
13265,6efce984242ca6456d74750810733a03,6ca46f2b9a1592929647682510800e0e,1,2018-08-31 09:51:47,2018-08-24 17:02:19
78242,6efce984242ca6456d74750810733a03,54282e97f61c23b78330c15b154c867d,1,2018-08-31 09:51:47,2018-09-03 09:06:57
41398,778826f9af6467ff6268bbcf436cd3c5,b58c726e8f5ba3196c593b1837d25744,1,2017-02-22 14:05:09,2017-02-13 15:54:42
92017,778826f9af6467ff6268bbcf436cd3c5,450c49623c365a4edcf0c5a2c93aa7c9,1,2017-02-22 14:05:09,2017-03-01 08:08:10
8813,a169c853a5ba10abfccb9c9afb06aa60,a3ef589d94a7dd3b117dc0e46c29d479,1,2018-03-11 00:15:42,2018-03-07 10:27:14
36453,a169c853a5ba10abfccb9c9afb06aa60,af07804a2d53e8639954baf97fbf803a,1,2018-03-11 00:15:42,2018-04-19 13:46:42
90864,a935b3d67f406bac4853776d80a05220,9db5ceb6f107ae0944c19188c8869a03,3,2018-05-08 02:06:50,2018-04-29 23:57:11


# Aggregate One-to-Many Order Attributes

In [12]:
payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_count=("payment_sequential", "count"),
        payment_type_count=("payment_type", "nunique"),
    )
    .reset_index()
)

primary_payment = (
    payments.loc[
        payments.groupby("order_id")["payment_value"].idxmax(),
        ["order_id", "payment_type"],
    ]
    .rename(columns={"payment_type": "primary_payment_type"})
)

payment_summary = payment_summary.merge(
    primary_payment,
    on="order_id",
    how="left",
    validate="one_to_one",
)

In [13]:
reviews_sorted = reviews.sort_values(
    ["order_id", "review_answer_timestamp"]
)

review_summary = (
    reviews_sorted.groupby("order_id")
    .agg(
        first_review_score=("review_score", "first"),
        avg_review_score=("review_score", "mean"),
        last_review_score=("review_score", "last"),
        review_count=("review_id", "size"),
        latest_review_answer=("review_answer_timestamp", "last"),
    )
    .reset_index()
)

In [14]:
assert payment_summary["order_id"].is_unique
assert review_summary["order_id"].is_unique

# Construct Order-Item Baseline

In [15]:
olist_df = (
    items
    .merge(
        orders,
        on="order_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        products,
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        sellers,
        on="seller_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        payment_summary,
        on="order_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        review_summary,
        on="order_id",
        how="left",
        validate="many_to_one",
    )
)

# Baseline Data Quality Audit

In [16]:
key_cols = [
    "order_id",
    "product_id",
    "seller_id",
    "customer_id",
    "customer_unique_id",
]

display(olist_df[key_cols].isna().sum())

order_id              0
product_id            0
seller_id             0
customer_id           0
customer_unique_id    0
dtype: int64

In [17]:
print(
    "Duplicate order-item keys:",
    olist_df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum(),
)

Duplicate order-item keys: 0


In [18]:
missing_summary = (
    olist_df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"] / len(olist_df) * 100
).round(2)

display(
    missing_summary[
        missing_summary["missing_count"] > 0
    ].sort_values("missing_pct", ascending=False)
)

,missing_count,missing_pct
order_delivered_customer_date,2454,2.18
product_category_name,1603,1.42
product_name_lenght,1603,1.42
product_description_lenght,1603,1.42
product_photos_qty,1603,1.42
order_delivered_carrier_date,1194,1.06
latest_review_answer,942,0.84
first_review_score,942,0.84
review_count,942,0.84
last_review_score,942,0.84


In [19]:
no_review_orders = (
    olist_df.loc[
        olist_df["review_count"].isna(),
        ["order_id", "order_status"],
    ]
    .drop_duplicates()
)

print("Orders without reviews:", len(no_review_orders))

display(
    no_review_orders["order_status"]
    .value_counts()
    .to_frame("orders")
)

Orders without reviews: 749


,orders
order_status,
delivered,646
shipped,75
canceled,17
processing,6
invoiced,5


In [20]:
# Fill null review counts with 0
olist_df["review_count"] = (
    olist_df["review_count"]
    .fillna(0)
    .astype(int)
)

**Missing values:** Remaining missing values reflect unavailable source attributes or lifecycle events that did not occur or were not recorded. These values are retained rather than globally imputed because appropriate treatment depends on the downstream analytical use. review_count is set to 0 when no review record exists because this represents an observed absence of review activity. Model-specific missing-value treatment will be performed during feature preparation as needed.

In [21]:
assert len(olist_df) == len(items)
assert olist_df[["order_id", "order_item_id"]].duplicated().sum() == 0

print("Final baseline shape:", olist_df.shape)
print("Unique orders:", olist_df["order_id"].nunique())

Final baseline shape: (112650, 39)
Unique orders: 98666


### Export consolidated dataset

In [22]:
olist_df.to_csv(
    "../01_data/preprocessed/olist_order_item_baseline.csv",
    index=False,
)